# Crear y Compilar la Red CNN-TCN Optimizada para TinyML y ESP32-S3 (1 Canal sEMG)

Este notebook define y compila la arquitectura convolucional temporal **CNN-TCN-SE (Temporal Convolutional Squeeze-and-Excitation)** adaptada para:
- **Restricción física:** 1 solo canal de entrada sEMG `(W, 1)` = (300, 1).
- **Paradigma TinyML / Edge AI:** Reemplazo de LSTM por convoluciones dilatadas separables (TCN), reduciendo el tamaño a `< 35 KB` en TFLite INT8 y ejecutando en `< 2 ms` sobre el ESP32-S3.
- **Optimizaciones de Grafo:** Estructura `Conv1D -> BatchNorm -> ReLU` que permite fusión al 100% por el conversor TFLite sin operaciones personalizadas.


In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf
import tensorflow_model_optimization as tfmot
from tensorflow.keras import layers, models
import os

# Función para buscar y cargar el archivo .env
def load_env_variables():
    from pathlib import Path
    try:
        start_dir = Path(os.getcwd())
    except:
        start_dir = Path(".")
        
    env_path = None
    for path in [start_dir] + list(start_dir.parents):
        temp_path = path / ".env"
        if temp_path.exists():
            env_path = temp_path
            break
            
    if env_path is None:
        raise FileNotFoundError("⚠️ No se pudo encontrar el archivo .env en la raíz del proyecto.")
        
    with open(env_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip()
            
    print(f"✅ Archivo .env cargado con éxito desde: {env_path}")

load_env_variables()

models_dir = os.environ["MODELS_DL_PROTO"]
os.makedirs(models_dir, exist_ok=True)


I0000 00:00:1786315901.475274   15984 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786315901.996461   15984 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786315904.517395   15984 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


✅ Archivo .env cargado con éxito desde: /home/cbe/Proyectos/MyoTensor_Tesis/.env


In [2]:
def squeeze_and_excitation_1d(input_tensor, ratio=4):
    """
    Bloque Squeeze-and-Excitation 1D para atención adaptativa de canales.
    Compatible al 100% con TFLite Micro (op GlobalAveragePooling1D + Dense).
    """
    filters = input_tensor.shape[-1]
    se = layers.GlobalAveragePooling1D()(input_tensor)
    se = layers.Dense(max(1, filters // ratio), activation='relu', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    se = layers.Reshape((1, filters))(se)
    return layers.multiply([input_tensor, se])

def tcn_se_block(input_tensor, filters, kernel_size=5, dilation_rate=1):
    """
    Bloque Convolucional Temporal Dilatado (TCN) con Convolución Separable y SE-1D.
    - SeparableConv1D para minimizar operaciones MACs.
    - Conv1D -> BatchNorm -> ReLU para fusión directa en INT8.
    - Conexión Residual si las dimensiones coinciden.
    """
    x = layers.SeparableConv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='same')(input_tensor)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = squeeze_and_excitation_1d(x)
    
    if input_tensor.shape[-1] == filters:
        x = layers.add([input_tensor, x])
    return x

def build_myotensor_1ch_tcn_model(input_shape=(300, 1), num_classes=4):
    """
    Arquitectura CNN-TCN-SE para 1 canal sEMG optimizada para TinyML / ESP32-S3.
    Reemplaza la LSTM para lograr máxima velocidad SIMD y mínima memoria.
    """
    inputs = layers.Input(shape=input_shape)
    
    # 1. Tallo convolucional inicial (reducción temporal)
    x = layers.Conv1D(16, kernel_size=7, strides=2, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.1)(x)
    
    # 2. Bloque TCN 1: Captura de transitorios sEMG rápidos (Dilation = 1)
    x = tcn_se_block(x, filters=24, kernel_size=5, dilation_rate=1)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)
    
    # 3. Bloque TCN 2: Captura de la envolvente muscular de medio/largo plazo (Dilation = 2)
    x = tcn_se_block(x, filters=32, kernel_size=5, dilation_rate=2)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.3)(x)
    
    # 4. Agregación temporal global sin capas recurrentes
    x = layers.GlobalAveragePooling1D()(x)
    
    # 5. Clasificador denso ultra-ligero
    x = layers.Dense(16)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)
    
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="MyoTensor_CNN_TCN_1Ch")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


In [3]:
model = build_myotensor_1ch_tcn_model((300, 1), 4)
model.summary()

model_path = os.path.join(models_dir, "myotensor_proto_net_tcn.keras")
print(f"💾 Guardando modelo CNN-TCN inicial en: {model_path} ...")
model.save(model_path)
print("🎉 ¡Arquitectura CNN-TCN para 1 canal guardada con éxito!")


W0000 00:00:1786315908.709949   15984 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "MyoTensor_CNN_TCN_1Ch"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 300, 1)]             0         []                            
                                                                                                  
 conv1d (Conv1D)             (None, 150, 16)              128       ['input_1[0][0]']             
                                                                                                  
 batch_normalization (Batch  (None, 150, 16)              64        ['conv1d[0][0]']              
 Normalization)                                                                                   
                                                                                                  
 activation (Activation)     (None, 150, 16)              0         ['batch_no